# 03 — Publication-Quality Visualizations

Generate all figures from cached results. Run this after Notebooks 01 and 02.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from src.utils import load_cached, ensure_dir
from src.visualization import (
    set_publication_style,
    plot_accuracy_comparison,
    plot_aggregate_trajectory,
    plot_logit_lens_heatmap,
    plot_divergence_onset_histogram,
    plot_fertility_vs_drop,
    plot_entropy_trajectories,
    plot_case_study,
    plot_small_multiples,
)

ensure_dir('../results/figures')
set_publication_style()

# Load cached results
data = load_cached('../results/cached_activations/analysis_results.pt')
behavioral = pd.read_csv('../results/tables/behavioral_results.csv')
fertility = pd.read_csv('../results/tables/fertility_analysis.csv')

print(f"Loaded {len(data['problems'])} problems")

In [ ]:
# Figure 1: Behavioral accuracy
acc_en = behavioral['correct_en'].mean() * 100
acc_tr = behavioral['correct_tr'].mean() * 100

# Build difficulty dict from behavioral data
difficulty_bins = {'2-step': (1, 2), '3-4 step': (3, 4), '5-6 step': (5, 6), '7+ step': (7, 100)}
acc_by_diff = {}
for label, (lo, hi) in difficulty_bins.items():
    mask = (behavioral['n_steps'] >= lo) & (behavioral['n_steps'] <= hi)
    subset = behavioral[mask]
    if len(subset) > 0:
        acc_by_diff[label] = {'en': subset['correct_en'].mean() * 100, 'tr': subset['correct_tr'].mean() * 100}

fig = plot_accuracy_comparison(acc_en, acc_tr, acc_by_diff, '../results/figures/fig1_accuracy_comparison.pdf')
fig.savefig('../results/figures/fig1_accuracy_comparison.png', dpi=300)
plt.show()

In [ ]:
# Figure 2: HERO FIGURE — Aggregate reasoning trajectory
fig = plot_aggregate_trajectory(
    data['mean_en'], data['ci_lo_en'], data['ci_hi_en'],
    data['mean_tr'], data['ci_lo_tr'], data['ci_hi_tr'],
    save_path='../results/figures/fig2_aggregate_trajectory.pdf',
)
fig.savefig('../results/figures/fig2_aggregate_trajectory.png', dpi=300)
plt.show()

In [ ]:
# Figure 3a/3b: Logit lens heatmaps for one example problem
# Pick the problem with the most dramatic divergence
divergences = [np.max(np.abs(pe - pt)) for pe, pt in zip(data['all_probs_en'], data['all_probs_tr'])]
best_idx = np.argmax(divergences)
p = data['problems'][best_idx]

fig_en = plot_logit_lens_heatmap(
    data['all_results_en'][best_idx]['top_tokens'],
    data['all_probs_en'][best_idx],
    f"English — Problem #{p['idx']} (answer={p['answer_number']})",
    save_path='../results/figures/fig3a_heatmap_en.pdf',
)
plt.show()

fig_tr = plot_logit_lens_heatmap(
    data['all_results_tr'][best_idx]['top_tokens'],
    data['all_probs_tr'][best_idx],
    f"Turkish — Problem #{p['idx']} (answer={p['answer_number']})",
    save_path='../results/figures/fig3b_heatmap_tr.pdf',
)
plt.show()

In [ ]:
# Figure 4: Divergence onset histogram
fig = plot_divergence_onset_histogram(
    data['onset_layers'],
    save_path='../results/figures/fig4_divergence_onset.pdf',
)
fig.savefig('../results/figures/fig4_divergence_onset.png', dpi=300)
plt.show()

In [ ]:
# Figure 5: Token fertility vs accuracy drop
fig = plot_fertility_vs_drop(
    fertility['fertility_ratio'].tolist(),
    behavioral['correct_en'].tolist(),
    behavioral['correct_tr'].tolist(),
    save_path='../results/figures/fig5_fertility_vs_drop.pdf',
)
fig.savefig('../results/figures/fig5_fertility_vs_drop.png', dpi=300)
plt.show()

In [ ]:
# Figure 6: Entropy trajectories
fig = plot_entropy_trajectories(
    data['mean_entropy_en'],
    data['mean_entropy_tr'],
    save_path='../results/figures/fig6_entropy_trajectories.pdf',
)
fig.savefig('../results/figures/fig6_entropy_trajectories.png', dpi=300)
plt.show()

In [ ]:
# Figure 7: Case study — two-panel deep dive
fig = plot_case_study(
    data['problems'][best_idx],
    data['all_probs_en'][best_idx],
    data['all_probs_tr'][best_idx],
    data['all_results_en'][best_idx]['top_tokens'],
    data['all_results_tr'][best_idx]['top_tokens'],
    save_path='../results/figures/fig7_case_study.pdf',
)
fig.savefig('../results/figures/fig7_case_study.png', dpi=300)
plt.show()

In [ ]:
# Figure 8: Small multiples — all 30 problems
fig = plot_small_multiples(
    data['all_probs_en'],
    data['all_probs_tr'],
    data['problems'],
    save_path='../results/figures/fig8_small_multiples.pdf',
)
fig.savefig('../results/figures/fig8_small_multiples.png', dpi=300)
plt.show()

print("\n=== All figures saved to results/figures/ ===")